In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [25]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

# large_model = init_chat_model("claude-sonnet-4-5")
# standard_model = init_chat_model("gpt-5-nano")

large_model = init_chat_model("gpt-5-nano")
standard_model = init_chat_model("gpt-4o")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [21]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [26]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

I haven't had a chance to water the office plant yet today. Do you know how often it needs to be watered, or should I check the care instructions to make sure it gets the right amount?


In [27]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-4o-2024-08-06


In [28]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

There isn’t a fixed calendar for replacing the pot. You’ll usually repot when the plant shows signs it’s outgrowing its container or the soil isn’t healthy anymore.

Common signs:
- Roots visible at or circling the bottom/through drainage holes (root-bound)
- Pot feels light or plant is top-heavy
- Soil dries out quickly after watering or drains poorly
- Slowed growth or leaves yellowing even with decent care
- Salt buildup or soil looks degraded after a while

Typical timing (rough guidelines):
- Small to medium office plants in ~4–6 inch pots: every 12–24 months
- Larger plants in bigger pots: every 2–4 years
- If you see signs sooner, repot earlier

Best time to repot:
- In spring or early summer, when the plant is actively growing

Choosing a new pot and soil:
- Go 1–2 inches larger in diameter than the current pot
- Ensure good drainage
- Use fresh, appropriate potting mix for the plant

Quick repot steps:
- Water the plant a day before
- Gently remove from the pot and tease out e

In [29]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07
